# Machine Learning Template

Use this notebook for leakage-aware prediction modeling and transparent reporting.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from dementia_ai_research.preprocessing import clean_dataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## Load and Clean Dataset

In [ ]:
# df_raw = pd.read_csv(PROJECT_ROOT / 'restricted_data' / 'analysis_dataset.csv')
# df = clean_dataset(df_raw)
df = pd.DataFrame()

## Define Outcome and Features

Exclude identifiers, post-outcome variables, free-text clinical notes, and variables that directly encode diagnostic labels unless justified.

In [ ]:
outcome_col = 'diagnosis_group'
positive_label = 'MCI'
numeric_features = ['age', 'education_years', 'mmse', 'moca']
categorical_features = ['sex']

if not df.empty:
    model_df = df.dropna(subset=[outcome_col]).copy()
    model_df['target'] = (model_df[outcome_col] == positive_label).astype(int)
    X = model_df[numeric_features + categorical_features]
    y = model_df['target']

## Build Baseline Pipeline

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

## Cross-Validated Evaluation

In [ ]:
if not df.empty:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    print(classification_report(y, y_pred))
    print({'roc_auc': roc_auc_score(y, y_proba)})